# PBMC CITE-seq F4 RNA covariate probe

This notebook is a focused RNA-only copy of `pbmc_citeseq_tutorial.ipynb` for testing the F4 covariate heads on the Hao et al. PBMC vaccination time-course dataset.

The dataset has three vaccination time points and repeated measurements from the same individuals. Here we use:

| Concept | Obs column | Meaning |
|---|---|---|
| group / condition | `time_point` | day 0, day 3, day 7 |
| donor / sample | `donor` | individual parsed from `orig.ident` |
| batch | `batch` | full donor-time identifier from `orig.ident` |
| biology label | `label_col` | preferred cell-type annotation |

The goal is diagnostic: train baseline and F4-enabled variants, then probe whether donor and batch signal move out of `z_shared` while cell-type signal remains available in `z_shared`. Formal multi-seed promotion gates belong in `scripts/benchmark_f4_covariate_probes.py`.

## 1. Environment

In [1]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "")

import warnings
warnings.filterwarnings("ignore")

from collections import OrderedDict

import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split

import spVIPESmulti

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
sc.settings.set_figure_params(dpi=90, frameon=False)

print(f"spVIPESmulti: {spVIPESmulti.__version__}")
print(f"scvi-tools  : {scvi.__version__}")
print(f"scanpy      : {sc.__version__}")
print(f"torch       : {torch.__version__} (CUDA: {torch.cuda.is_available()})")

spVIPESmulti: 1.0.0
scvi-tools  : 1.4.2
scanpy      : 1.12.1
torch       : 2.11.0+cu128 (CUDA: False)


## 2. Load PBMC CITE-seq RNA

The source dataset includes RNA and protein measurements, but this F4 probe intentionally keeps only the RNA matrix.

In [2]:
adata_full = scvi.data.pbmc_seurat_v4_cite_seq(save_path="docs/notebooks/data/")
adata_full.obs_names_make_unique()

# Remove this small population of cells that is not well annotated in the source tutorial.
adata_full = adata_full[adata_full.obs["celltype.l1"] != "other"].copy()

print("Shape       :", adata_full.shape)
print("Obs columns :", list(adata_full.obs.columns[:20]))
print()
print(adata_full.obs.head(3))

INFO     File docs/notebooks/data/pbmc_seurat_v4.h5ad already downloaded                                           
Shape       : (149753, 20729)
Obs columns : ['nCount_ADT', 'nFeature_ADT', 'nCount_RNA', 'nFeature_RNA', 'orig.ident', 'lane', 'donor', 'time', 'celltype.l1', 'celltype.l2', 'celltype.l3', 'Phase', 'nCount_SCT', 'nFeature_SCT', 'X_index', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'Protein log library size', 'Number proteins detected']

                     nCount_ADT  nFeature_ADT  nCount_RNA  nFeature_RNA  \
L1_AAACCCAAGAAACTCA      7430.0           221     10823.0          2915   
L1_AAACCCAAGACATACA      5949.0           211      5864.0          1617   
L1_AAACCCACAACTGGTT      6547.0           217      5067.0          1381   

                    orig.ident lane donor time celltype.l1 celltype.l2  ...  \
L1_AAACCCAAGAAACTCA       P2_7   L1    P2    7        Mono   CD14 Mono  ...   
L1_AAACCCAAGACATACA       P1_7   L1    P1    7       CD4 T     CD4 TCM  ...  

## 3. Parse F4 covariates

`orig.ident` encodes donor and time point, for example `P1_0`, `P1_3`, and `P1_7`. We parse donor from the prefix and use the full donor-time identifier as the batch covariate.

In [ ]:
label_col = "celltype.l1"
print(f"Cell-type label column: {label_col!r}")

print("Cells per time point:")
print(adata_full.obs["time"].value_counts().sort_index())

print("\nCells per donor:")
print(adata_full.obs["donor"].value_counts().sort_index())

print("\nCells per batch:")
print(adata_full.obs["lane"].value_counts().sort_index().head(12))

Cell-type label column: 'celltype.l1'
Cells per time point:
time
0    49658
3    50488
7    49607
Name: count, dtype: int64

Cells per donor:
donor
P1    17288
P2    16604
P3    14186
P4    16667
P5    19321
P6    18575
P7    23792
P8    23320
Name: count, dtype: int64

Cells per batch:
lane
E2L1     9812
E2L2    10952
E2L3    10386
E2L4    10518
E2L5    10877
E2L6    11035
E2L7    10826
E2L8    10602
L1      13341
L2      12728
L3      11684
L4      12938
Name: count, dtype: int64


In [6]:
donor_by_time = pd.crosstab(adata_full.obs["donor"], adata_full.obs["time"])
batch_by_time = pd.crosstab(adata_full.obs["lane"], adata_full.obs["time"])
timepoints_per_donor = donor_by_time.gt(0).sum(axis=1)

print("Donor by time point:")
display(donor_by_time)
print("\nBatches by time point (first 12 rows):")
display(batch_by_time.head(12))
print("\nNumber of time points represented per donor:")
print(timepoints_per_donor.sort_index())

Donor by time point:


time,0,3,7
donor,,,
P1,6076,5698,5514
P2,5744,5528,5332
P3,4535,4832,4819
P4,5178,5651,5838
P5,6186,6186,6949
P6,5505,6819,6251
P7,8369,7572,7851
P8,8065,8202,7053



Batches by time point (first 12 rows):


time,0,3,7
lane,,,
E2L1,3300,3280,3232
E2L2,3631,3669,3652
E2L3,3391,3502,3493
E2L4,3486,3536,3496
E2L5,3669,3664,3544
E2L6,3566,3758,3711
E2L7,3597,3774,3455
E2L8,3485,3596,3521
L1,4367,4548,4426



Number of time points represented per donor:
donor
P1    3
P2    3
P3    3
P4    3
P5    3
P6    3
P7    3
P8    3
dtype: int64


## 4. Subsample and select RNA HVGs

Defaults are chosen for a practical diagnostic run. Increase `N_PER_GROUP`, `N_HVG`, or `MAX_EPOCHS` below for a heavier run.

In [8]:
N_PER_GROUP = 10000
N_HVG = 5000

rng = np.random.default_rng(SEED)
time_col = adata_full.obs["time"].astype(str).to_numpy()
pos_keep = []
for tp in sorted(adata_full.obs["time"].astype(str).unique()):
    pos = np.where(time_col == tp)[0]
    pick = rng.choice(pos, size=min(N_PER_GROUP, len(pos)), replace=False)
    pos_keep.extend(pick.tolist())

adata = adata_full[np.array(sorted(pos_keep))].copy()
adata.obs_names_make_unique()

if "counts" in adata.layers:
    adata.X = adata.layers["counts"].copy()
    print("Using raw counts from adata.layers['counts']")

sc.pp.highly_variable_genes(adata, n_top_genes=N_HVG, flavor="seurat_v3", batch_key="lane")
adata = adata[:, adata.var["highly_variable"]].copy()

print(f"After subsample + HVG: {adata.shape}")
print("Cells per time point:")
print(adata.obs["time"].value_counts().sort_index())
print("\nDonor by time point after subsampling:")
display(pd.crosstab(adata.obs["donor"], adata.obs["time"]))

After subsample + HVG: (30000, 5000)
Cells per time point:
time
0    10000
3    10000
7    10000
Name: count, dtype: int64

Donor by time point after subsampling:


time,0,3,7
donor,,,
P1,1231,1133,1105
P2,1143,1042,1068
P3,896,996,976
P4,1074,1124,1164
P5,1220,1268,1384
P6,1124,1345,1254
P7,1675,1465,1639
P8,1637,1627,1410


## 5. Prepare spVIPESmulti input

We split the RNA AnnData into one group per time point and register the F4 covariates through `setup_anndata`.

In [9]:
time_points = sorted(adata.obs["time"].astype(str).unique())
adatas_dict = OrderedDict()
for tp in time_points:
    sub = adata[adata.obs["time"].astype(str) == tp].copy()
    sub.uns = {}
    sub.obsm = {}
    sub.layers = {}
    adatas_dict[tp] = sub
    print(f"{tp}: {sub.shape}")

adata_spv = spVIPESmulti.data.prepare_adatas(adatas_dict)

print(f"\nConcatenated AnnData: {adata_spv.shape}")
print(f"Groups: {list(adata_spv.uns['groups_mapping'].values())}")
print(f"Group sizes: {[len(g) for g in adata_spv.uns['groups_obs_indices']]}")

0: (10000, 5000)
3: (10000, 5000)
7: (10000, 5000)

Concatenated AnnData: (30000, 15000)
Groups: ['0', '3', '7']
Group sizes: [10000, 10000, 10000]


In [ ]:
spVIPESmulti.model.spVIPESmulti.setup_anndata(
    adata_spv,
    groups_key="time",
    label_key=label_col,
    sample_key="donor",
    condition_key="time",
    donor_key="donor",
    batch_key="lane",
)

group_indices_list = [list(map(int, g)) for g in adata_spv.uns["groups_obs_indices"]]
registered_obs = ["groups", label_col, "donor", "time_point", "batch"]
print("Registered obs columns:", registered_obs)
print(adata_spv.obs[registered_obs].head())

## 6. Train baseline and F4 variants

All variants start from `disentangle_preset="off"` so each row isolates the explicit F4 weights below. `use_nf_prior=False` keeps the run focused on the covariate heads.

In [ ]:
N_SHARED = 15
N_PRIVATE = 8
N_HIDDEN = 128
DROPOUT = 0.1
MAX_EPOCHS = 8
BATCH_SIZE = 512
KL_WARMUP = 4

F4_VARIANTS = OrderedDict({
    "baseline": {},
    "donor_private": {"disentangle_donor_private_weight": 0.5},
    "donor_shared": {"disentangle_donor_shared_weight": 0.5},
    "batch_shared": {"disentangle_batch_shared_weight": 0.5},
    "full_bio": {
        "disentangle_donor_shared_weight": 0.5,
        "disentangle_donor_private_weight": 0.5,
        "disentangle_batch_shared_weight": 0.5,
    },
})

MODEL_KWARGS = dict(
    n_hidden=N_HIDDEN,
    n_dimensions_shared=N_SHARED,
    n_dimensions_private=N_PRIVATE,
    dropout_rate=DROPOUT,
    use_nf_prior=False,
    disentangle_preset="off",
)

models = {}
latents_by_variant = {}
embed_keys = {}

In [ ]:
for variant, variant_kwargs in F4_VARIANTS.items():
    print(f"\n=== Training {variant} ===")
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    model = spVIPESmulti.model.spVIPESmulti(
        adata_spv,
        **MODEL_KWARGS,
        **variant_kwargs,
    )
    model.train(
        group_indices_list,
        batch_size=BATCH_SIZE,
        max_epochs=MAX_EPOCHS,
        train_size=0.9,
        early_stopping=True,
        n_epochs_kl_warmup=KL_WARMUP,
        accelerator="cpu",
        devices=1,
    )

    latents = model.get_latent_representation(group_indices_list, batch_size=BATCH_SIZE)
    prefix = f"X_spVIPESmulti_{variant}"
    spVIPESmulti.utils.store_latents(adata_spv, latents, group_indices_list, obsm_prefix=prefix)

    models[variant] = model
    latents_by_variant[variant] = latents
    embed_keys[variant] = {
        "shared": f"{prefix}_shared",
        "private": {g: f"{prefix}_private_g{g}" for g in range(len(group_indices_list))},
    }
    print("Stored:", embed_keys[variant])

## 7. Training diagnostics

The F4 losses are opt-in. When a variant enables a covariate head, its corresponding loss should appear in the training history.

In [ ]:
for variant, model in models.items():
    print(f"\n{variant}")
    history_keys = list(model.history.keys())
    print([k for k in history_keys if "disentangle" in k or "elbo" in k][:20])

    fig = spVIPESmulti.pl.training_curves(model)
    fig.suptitle(variant, y=1.02)
    plt.show()

## 8. Latent probe helpers

These probes ask how easily donor, batch, time point, and cell type can be predicted from each latent representation. Use them as diagnostics, not as formal gates.

In [ ]:
def _stitch_private(latents, group_indices_list, n_obs):
    """Create one full private matrix by placing each group's private latent in observation order."""
    sample = next(iter(latents["private_reordered"].values()))
    out = np.zeros((n_obs, sample.shape[1]), dtype=np.float32)
    for gi, idxs in enumerate(group_indices_list):
        out[np.asarray(idxs)] = latents["private_reordered"][gi]
    return out


def probe_balanced_accuracy(x, y, seed=SEED):
    y = np.asarray(y).astype(str)
    classes, counts = np.unique(y, return_counts=True)
    if classes.size < 2:
        return np.nan, "skipped: <2 classes"
    stratify = y if np.min(counts) >= 2 else None
    try:
        x_train, x_test, y_train, y_test = train_test_split(
            x,
            y,
            test_size=0.3,
            random_state=seed,
            stratify=stratify,
        )
        clf = LogisticRegression(max_iter=500, class_weight="balanced")
        clf.fit(x_train, y_train)
        pred = clf.predict(x_test)
        return float(balanced_accuracy_score(y_test, pred)), "ok"
    except Exception as exc:
        return np.nan, f"failed: {type(exc).__name__}: {exc}"


targets = OrderedDict({
    "donor": adata_spv.obs["donor"].values,
    "batch": adata_spv.obs["batch"].values,
    "time_point": adata_spv.obs["time_point"].values,
    "cell_type": adata_spv.obs[label_col].values,
})

In [ ]:
probe_rows = []
for variant, latents in latents_by_variant.items():
    latent_mats = {
        "shared": adata_spv.obsm[embed_keys[variant]["shared"]],
        "private": _stitch_private(latents, group_indices_list, adata_spv.n_obs),
    }
    for latent_name, latent_mat in latent_mats.items():
        for target_name, target_values in targets.items():
            score, note = probe_balanced_accuracy(latent_mat, target_values)
            probe_rows.append({
                "variant": variant,
                "latent": latent_name,
                "target": target_name,
                "balanced_accuracy": score,
                "notes": note,
            })

probe_df = pd.DataFrame(probe_rows)
probe_df.sort_values(["target", "latent", "variant"])

In [ ]:
summary_targets = [
    ("donor", "shared"),
    ("donor", "private"),
    ("batch", "shared"),
    ("cell_type", "shared"),
    ("time_point", "shared"),
]

summary_tables = {}
for target_name, latent_name in summary_targets:
    table = (
        probe_df.query("target == @target_name and latent == @latent_name")
        .set_index("variant")[["balanced_accuracy", "notes"]]
        .reindex(F4_VARIANTS.keys())
    )
    summary_tables[(target_name, latent_name)] = table
    print(f"\nTarget={target_name}, latent={latent_name}")
    display(table)

In [ ]:
plot_df = probe_df[
    probe_df.apply(
        lambda r: (r["target"], r["latent"]) in set(summary_targets),
        axis=1,
    )
].copy()
plot_df["probe"] = plot_df["target"] + " / " + plot_df["latent"]

fig, ax = plt.subplots(figsize=(12, 4))
sns.barplot(
    data=plot_df,
    x="variant",
    y="balanced_accuracy",
    hue="probe",
    ax=ax,
)
ax.set_ylim(0, 1)
ax.set_ylabel("Balanced accuracy")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=25)
ax.legend(title="Probe", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 9. Shared latent UMAP comparison

The key visual check is whether `full_bio` keeps cell-type structure while reducing donor and batch structure in the shared latent.

In [ ]:
for variant in ["baseline", "full_bio"]:
    spVIPESmulti.utils.compute_shared_umap(
        adata_spv,
        obsm_key=embed_keys[variant]["shared"],
        umap_key=f"X_umap_{variant}_shared",
        n_neighbors=15,
        min_dist=0.3,
    )

In [ ]:
for variant in ["baseline", "full_bio"]:
    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    for ax, color in zip(axes, [label_col, "time_point", "donor", "batch"]):
        sc.pl.embedding(
            adata_spv,
            basis=f"X_umap_{variant}_shared",
            color=color,
            ax=ax,
            show=False,
            title=f"{variant}: {color}",
            frameon=False,
        )
    plt.tight_layout()
    plt.show()

## 10. Private latent UMAP comparison

Private latents are group-specific. These plots inspect whether donor and batch structure remains visible in private space for each time point.

In [ ]:
private_group_adatas = {}
for variant in ["baseline", "full_bio"]:
    private_group_adatas[variant] = OrderedDict()
    for gi, tp in enumerate(time_points):
        mask = adata_spv.obs["groups"].astype(str).to_numpy() == tp
        sub = adata_spv[mask].copy()
        key = embed_keys[variant]["private"][gi]
        sub.obsm[f"X_{variant}_private"] = adata_spv.obsm[key][mask]
        sc.pp.neighbors(sub, use_rep=f"X_{variant}_private", key_added=f"{variant}_private_nn")
        sc.tl.umap(sub, neighbors_key=f"{variant}_private_nn", min_dist=0.5)
        sub.obsm[f"X_umap_{variant}_private"] = sub.obsm["X_umap"].copy()
        private_group_adatas[variant][tp] = sub

In [ ]:
for variant in ["baseline", "full_bio"]:
    for color in ["donor", "batch"]:
        fig, axes = plt.subplots(1, len(time_points), figsize=(5 * len(time_points), 4))
        if len(time_points) == 1:
            axes = [axes]
        for ax, tp in zip(axes, time_points):
            sc.pl.embedding(
                private_group_adatas[variant][tp],
                basis=f"X_umap_{variant}_private",
                color=color,
                ax=ax,
                show=False,
                title=f"{variant} {tp}: {color}",
                frameon=False,
            )
        plt.tight_layout()
        plt.show()

## 11. Integration metrics

These metrics complement the probes. They should not be read alone: F4 should reduce unwanted donor/batch predictability without destroying cell-type structure.

In [ ]:
metric_rows = []
for variant in F4_VARIANTS:
    rep = adata_spv.obsm[embed_keys[variant]["shared"]]
    metric_rows.append({
        "variant": variant,
        "kBET_time_point_higher_better": spVIPESmulti.metrics.kbet(rep, adata_spv.obs["time_point"].values),
        "kNN_purity_cell_type_higher_better": spVIPESmulti.metrics.knn_purity(rep, adata_spv.obs[label_col].values),
        "iLISI_time_point_higher_better": spVIPESmulti.metrics.ilisi(rep, adata_spv.obs["time_point"].values, k=30),
        "ARI_cell_type_higher_better": spVIPESmulti.metrics.leiden_ari(rep, adata_spv.obs[label_col].values),
    })

metrics_df = pd.DataFrame(metric_rows).set_index("variant")
metrics_df

## 12. Interpretation checklist

Use this notebook to inspect whether the qualitative F4 behavior matches the intended direction:

- `donor_private`: donor probe accuracy on `z_private` should be retained or increased relative to baseline.
- `donor_shared`: donor probe accuracy on `z_shared` should drop relative to baseline.
- `batch_shared`: batch probe accuracy on `z_shared` should drop relative to baseline.
- `full_bio`: should combine donor-private retention with donor and batch erasure from `z_shared`.
- Cell-type probe accuracy and cell-type UMAP structure in `z_shared` should remain close to baseline.

Caveat: `batch = orig.ident` is nested within donor-time combinations, so batch probes are partly confounded with donor and time-point effects. Treat batch results here as diagnostic, not as definitive proof of batch disentanglement.